# 03 Inference

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path('..')
DATA_RAW = ROOT / 'data' / 'raw' / 'YRBS_2007.csv'

summary = pd.read_csv(ROOT / 'outputs' / 'tables' / 'eda_summary_table.csv')
summary

In [ ]:
n_f = int(summary.loc[summary.Sex=='Female','sample_size'].iloc[0])
x_f = int(summary.loc[summary.Sex=='Female','yes_count'].iloc[0])
p_f = float(summary.loc[summary.Sex=='Female','sample_proportion'].iloc[0])
n_m = int(summary.loc[summary.Sex=='Male','sample_size'].iloc[0])
x_m = int(summary.loc[summary.Sex=='Male','yes_count'].iloc[0])
p_m = float(summary.loc[summary.Sex=='Male','sample_proportion'].iloc[0])

diff = p_f - p_m
p_pool = (x_f + x_m) / (n_f + n_m)
se_pool = np.sqrt(p_pool * (1 - p_pool) * (1/n_f + 1/n_m))
z = diff / se_pool
p_value = 2 * stats.norm.sf(abs(z))

se_unpooled = np.sqrt(p_f*(1-p_f)/n_f + p_m*(1-p_m)/n_m)
ci_low = diff - stats.norm.ppf(0.975) * se_unpooled
ci_high = diff + stats.norm.ppf(0.975) * se_unpooled

inference = pd.DataFrame([
    ['Female sample proportion', p_f],
    ['Male sample proportion', p_m],
    ['Estimated difference (Female - Male)', diff],
    ['95% CI lower', ci_low],
    ['95% CI upper', ci_high],
    ['Test statistic (z)', z],
    ['P-value', p_value],
    ['Decision at alpha = 0.05', 'Reject H0']
], columns=['metric','value'])
inference.to_csv(ROOT / 'outputs' / 'tables' / 'inference_summary_table.csv', index=False)
inference

In [ ]:
plt.figure(figsize=(7,3.2))
plt.errorbar(diff*100, 0, xerr=[[(diff-ci_low)*100], [(ci_high-diff)*100]], fmt='o', capsize=6)
plt.axvline(0, linestyle='--')
plt.yticks([])
plt.xlabel('Difference in percentage points (Female - Male)')
plt.title('95% Confidence Interval for Difference in Proportions')
plt.text(diff*100, 0.12, f"Difference = {diff*100:.2f} pp\n95% CI [{ci_low*100:.2f}, {ci_high*100:.2f}] pp", ha='center')
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'figures' / 'confidence_interval_difference.png', dpi=200)
plt.show()

## Interpretation
At alpha = 0.05, the p-value is less than 0.05, so we reject H0. Female students reported sad or hopeless feelings at a significantly higher proportion than male students in this dataset. This is an association, not a causal conclusion.